# Sizing the dsm2_osc Ring Oscillator

Adapted from `macros/sw_tgate_cmfb2/scripts/sizing/size_sw_tgate_cmfb2.ipynb`, which in turn follows `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, the overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

`dsm2_osc` was sized by hand and by simulation sweeps. The unit finger W 0.5u L 0.5u came from the packing of the layout and the frequency was set with the node caps and the `MD` multiplier. This notebook reconstructs the derivation after the fact: from the spec to the sizes, then it reads the drawn sizes from the schematic and compares them.

Every number below is a gm/ID table model estimate, not a simulation and not a measurement. The simulated frequencies quoted from the README are single runs of the schematic and are marked as simulation.

The delay model is the simplest one that fits: each stage is a capacitance `C` charged and discharged by its tail current `I` over half the supply, so `t_d = C * VDD / (2 * I)` and the ring of `N` stages oscillates at `f_ring = 1 / (2 * N * t_d)`. The flip-flop divides by 2, so the clock is `f_ring / 2`. It ignores the trip point of the inverter, the Miller effect on the inverter gate caps, the layout wiring capacitance, the slope dependence of the delay and the current the inverter itself can supply. The README reports the extracted view about 21 % slower than the schematic, which is the wiring capacitance this model leaves out.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

### Spec

From the README. The bank current is `ibias_in = 8.33 nA * (16 + 2 * code)`. The target is 1.024 MHz at code 4 for `dsm2_core`. The ring has 5 starved stages and the clock is divided by 2.

In [2]:
# Supply, bank current, target and ring topology
VDD      = 1.5
UNIT     = 8.33e-9            # A, one bank unit
CODES    = (0, 4, 15)
N        = 5                  # ring stages
DIV      = 2                  # flip-flop divider
F_TARGET = 1.024e6            # Hz at code 4
PAD_V    = (0.3, 0.8)         # V, ibias_pad range of the pass switch
CELL     = 'dsm2_osc'
DATA     = Path('../../../macros/inverter/scripts/sizing/data')
SCH      = Path('../schematic') / (CELL + '.sch')

# Simulated clock in kHz from the README (tt 1.5 V 27 C), and the spread of code 4 over all 27 conditions
SIM_TT   = {0: 565.9, 4: 822.9, 15: 1479.1}
SIM_C4   = (733.0, 840.0)     # ff 1.55 V 125 C, ss 1.45 V -40 C ... tt 1.45 V

def ibias_in(code):
    return UNIT * (16 + 2 * code)

### Read the sizes out of the schematic

The schematic is the source of truth. The parser reads `w`, `l` and `m` of every `sg13_*` transistor by instance name. The inverter and the tails carry names with the stage number appended (`XTP1` to `XTP5`), so the stages are collected by prefix. The checks are the invariants the topology rests on: the five stages are identical, the tails are the unit finger and the clamps share the inverter length.

In [3]:
def num(s):
    return float(s.rstrip('u'))

def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'C \{(sg13_(?:lv|hv)_[np]mos)\.sym\} .*?\{name=(\w+)\n(.*?)\n\}', txt, re.S):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(3), re.M))
        dev[m.group(2)] = dict(kind=m.group(1), w=num(a['w']), l=num(a['l']), m=int(a['m']))
    def stages(prefix):
        grp = [dev[f'{prefix}{i}'] for i in range(1, N + 1)]
        assert all(g == grp[0] for g in grp), prefix + ' differs between stages'
        return grp[0]
    ring = {p: stages(p) for p in ('XTP', 'XP', 'XN', 'XTN', 'XC')}
    ring['XK_N'] = stages('XK') if False else dev['XK1']
    assert dev['XK3'] == dev['XK5'] == dev['XK1'] and dev['XK2'] == dev['XK4']
    ring['XK_P'] = dev['XK2']
    return dev, ring

DEV, RING = read_sizes(SCH)
for k, v in RING.items():
    print(f"{k:5s} {v['kind']:12s} W {v['w']:g} um  L {v['l']:g} um  m {v['m']}")
for k in ('MD', 'M2', 'MPD', 'MGD', 'MPK', 'MSP', 'MSN'):
    v = DEV[k]
    print(f"{k:5s} {v['kind']:12s} W {v['w']:g} um  L {v['l']:g} um  m {v['m']}")
sch = [DEV[f'XS{t}{i}'] for t in 'PN' for i in (1, 2, 3)]
print('Schmitt devices', {(d['w'], d['l'], d['m']) for d in sch})

XTP   sg13_lv_pmos W 0.5 um  L 0.5 um  m 1
XP    sg13_lv_pmos W 1 um  L 0.13 um  m 1
XN    sg13_lv_nmos W 1 um  L 0.13 um  m 1
XTN   sg13_lv_nmos W 0.5 um  L 0.5 um  m 1
XC    sg13_lv_nmos W 0.5 um  L 0.5 um  m 1
XK_N  sg13_lv_nmos W 0.5 um  L 0.13 um  m 1
XK_P  sg13_lv_pmos W 0.5 um  L 0.13 um  m 1
MD    sg13_lv_nmos W 0.5 um  L 0.5 um  m 2
M2    sg13_lv_nmos W 0.5 um  L 0.5 um  m 1
MPD   sg13_lv_pmos W 0.5 um  L 0.5 um  m 1
MGD   sg13_lv_nmos W 0.5 um  L 0.5 um  m 1
MPK   sg13_lv_pmos W 0.5 um  L 0.5 um  m 1
MSP   sg13_lv_pmos W 1 um  L 0.5 um  m 2
MSN   sg13_hv_nmos W 1 um  L 0.45 um  m 1
Schmitt devices {(0.5, 0.5, 1)}


### Lookups

Four tables: LV NMOS and PMOS for the ring, mirrors and Schmitt trigger, HV NMOS for the branch switch `MSN`. All values are scaled from the width of the table to the drawn width.

In [4]:
nmos = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pmos = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
hvn  = lk(str(DATA / 'sg13cmos5l_hv_nmos.mat'))

def get(t, par, w, **kw):
    # table value scaled to drawn width w in um
    return float(np.squeeze(t.lookup(par, **kw))) * w / t['W']

UF = RING['XTN']              # unit finger
LU, WU = UF['l'], UF['w']

### Bias mirror and tail current

`ibias_in` enters the diode `MD`. `M2` copies it into the PMOS diode `MPD`, which sets `vbp`. Each NMOS tail `XTN` mirrors `MD`, each PMOS tail `XTP` mirrors `MPD`. With all devices at the same W and L the ideal currents are set by the multipliers alone.

In [5]:
r_m2  = DEV['M2']['m']  / DEV['MD']['m']
r_tn  = RING['XTN']['m'] / DEV['MD']['m']
r_tp  = RING['XTP']['m'] / DEV['MPD']['m'] * r_m2
print(f"M2/MD = {r_m2:g}, XTN/MD = {r_tn:g}, XTP/MPD*M2/MD = {r_tp:g}")
print(f"{'code':>4} {'Ibias_in':>9} {'I_tail n':>9} {'I_tail p':>9} {'ring total':>11}   (nA)")
for c in CODES:
    i = ibias_in(c)
    print(f"{c:4d} {i*1e9:9.1f} {i*r_tn*1e9:9.1f} {i*r_tp*1e9:9.1f} {N*i*r_tn*1e9:11.1f}")

M2/MD = 0.5, XTN/MD = 0.5, XTP/MPD*M2/MD = 0.5
code  Ibias_in  I_tail n  I_tail p  ring total   (nA)
   0     133.3      66.6      66.6       333.2
   4     199.9     100.0     100.0       499.8
  15     383.2     191.6     191.6       958.0


### Tail gm/ID and headroom

The tails are the unit finger at 0.5u / 0.5u. The diode `MD` at `m` fingers carries `ibias_in`, so its gate voltage follows from the table with `ID = ibias_in / m` per finger. The same `VGS` on the tail gives gm/ID and `VDSS`. The tail has the stage's other three devices in series, so the swing at the inverter source is what it leaves. The drain voltage dependence is the ratio of the current at the tail's own `VDS` to the current at the diode's `VDS = VGS`.

In [6]:
VG = np.arange(0.05, 0.9, 0.005)
def vgs_for(t, i_finger):
    # gate voltage of a diode connected finger carrying i_finger
    idv = np.array([get(t, 'ID', WU, L=LU, VGS=v, VDS=v, VSB=0) for v in VG])
    return float(np.interp(i_finger, idv, VG))

VDS_TAIL = 0.4                # V, assumed tail drain voltage in the ring
RATIO = {}                    # code -> (NMOS tail over MD, M2 over MD) at the drain voltages above
print(f"{'code':>4} {'VGS_n':>6} {'gm/ID':>6} {'VDSS':>6} {'ID(0.4)/ID(diode)':>18} | {'VSG_p':>6} {'gm/ID':>6} {'VDSS':>6} {'M2 VDS':>7} {'ID ratio':>9} {'tail p':>7}")
for c in CODES:
    i = ibias_in(c)
    vn = vgs_for(nmos, i / DEV['MD']['m'])
    gn = float(np.squeeze(nmos.lookup('GM_ID', L=LU, VGS=vn, VDS=VDS_TAIL, VSB=0)))
    dn = float(np.squeeze(nmos.lookup('VDSS', L=LU, VGS=vn, VDS=VDS_TAIL, VSB=0)))
    rn = get(nmos, 'ID', WU, L=LU, VGS=vn, VDS=VDS_TAIL, VSB=0) / get(nmos, 'ID', WU, L=LU, VGS=vn, VDS=vn, VSB=0)
    ip = i * r_m2 / DEV['MPD']['m']
    vp = vgs_for(pmos, ip)
    gp = float(np.squeeze(pmos.lookup('GM_ID', L=LU, VGS=vp, VDS=VDS_TAIL, VSB=0)))
    dp = float(np.squeeze(pmos.lookup('VDSS', L=LU, VGS=vp, VDS=VDS_TAIL, VSB=0)))
    # M2 drains into the PMOS diode, so it sits at VDS = VDD - VSG_p instead of VGS_n
    vm2 = VDD - vp
    r2 = get(nmos, 'ID', WU, L=LU, VGS=vn, VDS=vm2, VSB=0) / get(nmos, 'ID', WU, L=LU, VGS=vn, VDS=vn, VSB=0)
    rp = get(pmos, 'ID', WU, L=LU, VGS=vp, VDS=VDS_TAIL, VSB=0) / get(pmos, 'ID', WU, L=LU, VGS=vp, VDS=vp, VSB=0)
    RATIO[c] = (rn, r2, rp)
    print(f"{c:4d} {vn:6.3f} {gn:6.1f} {dn:6.3f} {rn:18.3f} | {vp:6.3f} {gp:6.1f} {dp:6.3f} {vm2:7.3f} {r2:9.3f} {rp:7.3f}")

code  VGS_n  gm/ID   VDSS  ID(0.4)/ID(diode) |  VSG_p  gm/ID   VDSS  M2 VDS  ID ratio  tail p
   0  0.212   28.4  0.112              1.286 |  0.352   25.1  0.113   1.148     2.247   1.007


   4  0.227   27.3  0.113              1.252 |  0.368   23.3  0.116   1.132     2.153   1.005


  15  0.251   25.0  0.115              1.198 |  0.399   19.7  0.124   1.101     1.984   1.000


The tails run in weak inversion. The ratios in the table are the drain voltage effect the README mentions: the NMOS tail at 0.4 V carries more than the diode copy at `VDS = VGS`, and `M2` at the higher drain voltage of the PMOS diode carries more than `MD`, which pushes the PMOS tails up as well. These are DIBL and channel length modulation at `L = 0.5u` and they set the upper bound for the frequency below. The tail gm/ID is the weak inversion value, so the tail is a current source with `VDSS` of about 0.1 V and the inverter gets close to both rails.

### Node capacitance per stage

Node `n_i` is the output of stage `i`. It carries the gate of the next inverter (`XP` plus `XN` of the following stage, with `n5` also driving the four Schmitt input gates), the NMOS cap `XC` with drain, source and bulk on `vss`, the drain caps of `XP` and `XN` of its own stage and the drain of the clamp. Gate caps are averaged over `VGS` from 0 to `VDD` because the node swings through the whole range. Drain caps are taken at `VDS = VDD / 2`.

In [7]:
VGSW = np.linspace(0, VDD, 7)
def cgg_avg(t, w, l, vds):
    return np.mean([get(t, 'CGG', w, L=l, VGS=v, VDS=vds, VSB=0) for v in VGSW])

def cap_node(ring, schmitt=False, vds_cap=0.0):
    xn, xp, xc = ring['XN'], ring['XP'], ring['XC']
    kn, kp = ring['XK_N'], ring['XK_P']
    comp = {}
    comp['next inverter gates'] = cgg_avg(nmos, xn['w'] * xn['m'], xn['l'], VDD / 2) + cgg_avg(pmos, xp['w'] * xp['m'], xp['l'], VDD / 2)
    comp['XC node cap'] = cgg_avg(nmos, xc['w'] * xc['m'], xc['l'], vds_cap)
    comp['XN, XP drains'] = get(nmos, 'CDD', xn['w'] * xn['m'], L=xn['l'], VGS=0, VDS=VDD / 2, VSB=0) + get(pmos, 'CDD', xp['w'] * xp['m'], L=xp['l'], VGS=0, VDS=VDD / 2, VSB=0)
    comp['clamp drain'] = get(nmos, 'CDD', kn['w'], L=kn['l'], VGS=0, VDS=VDD / 2, VSB=0)
    if schmitt:
        comp['Schmitt gates'] = 2 * cgg_avg(nmos, WU, LU, VDD / 2) + 2 * cgg_avg(pmos, WU, LU, VDD / 2)
    return comp

def report(ring, label, **kw):
    comp = cap_node(ring, **kw)
    tot = sum(comp.values())
    print(label)
    for k, v in comp.items():
        print(f"  {k:20s} {v*1e15:6.2f} fF")
    print(f"  {'total':20s} {tot*1e15:6.2f} fF")
    return tot

C_STD = report(RING, 'node n1 to n4')
C_N5  = report(RING, 'node n5', schmitt=True)
C_AVG = (4 * C_STD + C_N5) / N
print(f"mean over the ring {C_AVG*1e15:.2f} fF")

node n1 to n4
  next inverter gates    2.96 fF
  XC node cap            2.65 fF
  XN, XP drains          1.58 fF
  clamp drain            0.44 fF
  total                  7.62 fF
node n5
  next inverter gates    2.96 fF
  XC node cap            2.65 fF
  XN, XP drains          1.58 fF
  clamp drain            0.44 fF
  Schmitt gates          8.47 fF
  total                 16.10 fF
mean over the ring 9.32 fF


### Frequency of the cell as drawn

With the tail currents of the mirror and the mean node capacitance, `f_ring = I / (N * C * VDD)` and the clock is that divided by 2. The ring current in the first column is the NMOS tail current. The last columns are the simulation from the README and the model over simulation. The slope of the stages differs between node 5 and the others, so the mean capacitance is used, which is exact only if all stages see the same current, as they do here.

In [8]:
def f_clk(i_tail, c):
    td = c * VDD / 2 / i_tail
    return 1 / (2 * N * td) / DIV

print(f"{'code':>4} {'I_tail (nA)':>12} {'t_d (ns)':>9} {'f_ring (MHz)':>13} {'f_clk (kHz)':>12} {'sim (kHz)':>10} {'model/sim':>10}")
f_mod = {}
for c in CODES:
    it = ibias_in(c) * r_tn
    f_mod[c] = f_clk(it, C_AVG)
    td = C_AVG * VDD / 2 / it
    print(f"{c:4d} {it*1e9:12.1f} {td*1e9:9.1f} {DIV*f_mod[c]/1e6:13.3f} {f_mod[c]/1e3:12.1f} {SIM_TT[c]:10.1f} {f_mod[c]/1e3/SIM_TT[c]:10.2f}")
print()
print('upper bound with the drain voltage ratios of the tail table (tail VDS 0.4 V), mean of the NMOS and PMOS tail:')
f_cor = {}
for c in CODES:
    rn, r2, rp = RATIO[c]
    it = ibias_in(c) * r_tn * 0.5 * (rn + r2 * rp)
    f_cor[c] = f_clk(it, C_AVG)
    print(f"{c:4d} I_tail {it*1e9:7.1f} nA  f_clk {f_cor[c]/1e3:7.1f} kHz  model/sim {f_cor[c]/1e3/SIM_TT[c]:.2f}")
print()
print(f"target {F_TARGET/1e3:.0f} kHz at code 4: model is {100*(f_mod[4]/F_TARGET-1):+.1f} %, simulation {100*(SIM_TT[4]*1e3/F_TARGET-1):+.1f} %")
print(f"range over codes, model: {f_mod[0]/1e3:.0f} to {f_mod[15]/1e3:.0f} kHz, tt simulation: {SIM_TT[0]:.0f} to {SIM_TT[15]:.0f} kHz")
print(f"code 4 over all 27 conditions, simulation: {SIM_C4[0]:.0f} to {SIM_C4[1]:.0f} kHz")

code  I_tail (nA)  t_d (ns)  f_ring (MHz)  f_clk (kHz)  sim (kHz)  model/sim
   0         66.6     104.9         0.954        476.8      565.9       0.84
   4        100.0      69.9         1.430        715.2      822.9       0.87
  15        191.6      36.5         2.742       1370.8     1479.1       0.93

upper bound with the drain voltage ratios of the tail table (tail VDS 0.4 V), mean of the NMOS and PMOS tail:
   0 I_tail   118.3 nA  f_clk   846.1 kHz  model/sim 1.50
   4 I_tail   170.8 nA  f_clk  1221.9 kHz  model/sim 1.48
  15 I_tail   304.9 nA  f_clk  2181.6 kHz  model/sim 1.47

target 1024 kHz at code 4: model is -30.2 %, simulation -19.6 %
range over codes, model: 477 to 1371 kHz, tt simulation: 566 to 1479 kHz
code 4 over all 27 conditions, simulation: 733 to 840 kHz


The model with ideal mirror currents puts code 4 at 715 kHz, 30 % below the target, so the drawn sizes miss the target in the model as well and the miss is not only a simulation effect. The simulation is 20 % below. The model is linear in the bank current. The simulated frequency grows less than linearly (1 : 1.45 : 2.61 against 1 : 1.5 : 2.9), which this model cannot show because it has no inverter drive limit. With the drain voltage ratios of the tail table the model moves up to about 1.2 MHz at code 4, 1.5 times the simulation. That is an upper bound, because the tail drain sits at 0.4 V only if the inverter drops nothing, so the simulated 823 kHz lies between the two model values. The ideal mirror alone is therefore not the cause of the miss, the current ratio uncertainty is as large as the miss. The three biggest items left out are the Miller effect on the inverter gates (raises the capacitance), the wiring capacitance and the real trip point.

### Sizing for a target

Two knobs give the target at code 4: the node capacitance through `XC`, or the tail current through the `MD` multiplier. Both come from the model, then scaled by the simulated over modelled frequency at code 4 so the answer is the one that would land the simulated clock on the target. The scaled answer is a first guess for a sweep, not a result.

In [9]:
it4 = ibias_in(4) * r_tn
C_REQ = it4 / (N * VDD * F_TARGET * DIV)
k_sim = SIM_TT[4] * 1e3 / f_mod[4]
C_REQ_S = C_REQ * k_sim                     # cap that lands the simulated clock on the target
xc = RING['XC']
c_xc = cgg_avg(nmos, xc['w'] * xc['m'], xc['l'], 0.0)
c_per_um2 = c_xc / (xc['w'] * xc['l'])
other = C_AVG - c_xc
print(f"node cap for the target, model          {C_REQ*1e15:5.2f} fF (drawn {C_AVG*1e15:.2f} fF)")
print(f"node cap for the target, sim scaled     {C_REQ_S*1e15:5.2f} fF")
print(f"XC gate cap density {c_per_um2*1e15:.2f} fF/um^2, rest of the node {other*1e15:.2f} fF")
for lab, c in (('model', C_REQ), ('sim scaled', C_REQ_S)):
    a = (c - other) / c_per_um2
    print(f"  XC area, {lab:10s} {a:5.3f} um^2 (drawn {xc['w']*xc['l']:.3f} um^2, so W {a/xc['l']:.2f} um at L {xc['l']} um)")
md_req = DEV['MD']['m'] * f_mod[4] / F_TARGET
print(f"MD multiplier for the target at drawn caps, model: {md_req:.2f} for the drawn {DEV['MD']['m']} (fewer fingers mean more current)")
print(f"MD multiplier, sim scaled: {DEV['MD']['m'] * SIM_TT[4] * 1e3 / F_TARGET:.2f}, a continuous number, the sheet has m = 2 or m = 1")

node cap for the target, model           6.51 fF (drawn 9.32 fF)
node cap for the target, sim scaled      7.49 fF
XC gate cap density 10.58 fF/um^2, rest of the node 6.67 fF
  XC area, model      -0.015 um^2 (drawn 0.250 um^2, so W -0.03 um at L 0.5 um)
  XC area, sim scaled 0.077 um^2 (drawn 0.250 um^2, so W 0.15 um at L 0.5 um)
MD multiplier for the target at drawn caps, model: 1.40 for the drawn 2 (fewer fingers mean more current)
MD multiplier, sim scaled: 1.61, a continuous number, the sheet has m = 2 or m = 1


The ideal-current model asks for 6.5 fF per node and the drawn node is 9.3 fF, of which 6.7 fF are not `XC`. With `XC` removed the node would still be above the model requirement, so the cap alone cannot reach the target in the model, and the tails would have to carry about 1.4 times the ideal current, which the drain voltage effect alone could supply. The sim scaled answer needs `XC` at 0.077 um^2, about a third of the drawn finger, or `MD` m1.6. `MD` m1 would double the tail currents and land the clock near 1.4 to 1.6 MHz, 40 to 60 % too high, and m2 is the only whole number that is below the target. The 20 % miss in the README is the consequence of that quantisation plus the 5 fF Schmitt load on `n5`. This is reported here as a disagreement between the derivation and the drawn cell, nothing is changed.

### Highest usable clock

From the README: the `sc_int2` OTA settles with 16.2 ns, `clk_noverlap` leaves 3.8 to 4.9 ns, and 10 time constants per phase plus 5 ns give about 2.9 MHz. The modelled code 15 clock is checked against that.

In [10]:
TAU, GAP, NTAU = 16.2e-9, 5e-9, 10
f_max = 1 / (2 * (NTAU * TAU + GAP))
print(f"f_max from sc_int2 settling {f_max/1e6:.2f} MHz")
print(f"code 15, model {f_mod[15]/1e6:.2f} MHz, margin {f_max/f_mod[15]:.1f}x")
print(f"code 15, simulation tt {SIM_TT[15]/1e3:.2f} MHz, fastest corner in the README 1.52 MHz, margin {f_max/1.52e6:.1f}x")

f_max from sc_int2 settling 2.99 MHz
code 15, model 1.37 MHz, margin 2.2x
code 15, simulation tt 1.48 MHz, fastest corner in the README 1.52 MHz, margin 2.0x


### Schmitt trigger thresholds

The restorer is the classic 6T. `XSP1`, `XSP2`, `XSN1` and `XSN2` have their gates on `n5`. The feedback devices `XSN3` (drain `vdd`, source `sb`, the mid node of the NMOS stack) and `XSP3` (drain `vss`, source `sa`) have their gates on the output `r1`.

Rising input, output still high: `XSN3` holds `sb` up and `XSN1` pulls it down, so `sb` settles where the two currents are equal. The output only starts to fall when `XSN2` turns on, which needs `vin - sb` to reach its threshold voltage, so that `vin` is the upper threshold. The falling threshold is the mirror image with the PMOS devices. It is a one-point estimate. The regeneration and the hysteresis window the output sees are not modelled, and all six devices are unit fingers, so this is also the case where the feedback is as strong as the stack.

In [11]:
def solve_mid(t, u):
    # mid node x (measured from the stack's own rail) where the feedback current equals the stack current
    f = lambda x: (get(t, 'ID', WU, L=LU, VGS=VDD - x, VDS=VDD - x, VSB=x)
                   - get(t, 'ID', WU, L=LU, VGS=u, VDS=x, VSB=0))
    lo, hi = 0.05, VDD - 0.05
    for _ in range(40):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (f(lo) > 0):
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def threshold(t):
    # input u, measured from the stack's rail, where the upper device of the stack starts to conduct
    for u in np.arange(0.3, 1.25, 0.01):
        x = solve_mid(t, u)
        vt = float(np.squeeze(t.lookup('VTH', L=LU, VGS=u - x, VDS=VDD - x, VSB=x)))
        if u - x >= vt:
            return u, x, vt

u_n, sb, vtn = threshold(nmos)
u_p, sa, vtp = threshold(pmos)
vt_hi, vt_lo = u_n, VDD - u_p
print(f"upper threshold {vt_hi:.2f} V (sb {sb:.2f} V, Vth of XSN2 {vtn:.2f} V)")
print(f"lower threshold {vt_lo:.2f} V (sa {sa:.2f} V below vdd, Vth of XSP2 {vtp:.2f} V)")
print(f"hysteresis {vt_hi - vt_lo:.2f} V, margin to the rails {VDD - vt_hi:.2f} V above and {vt_lo:.2f} V below")

upper threshold 0.90 V (sb 0.56 V, Vth of XSN2 0.33 V)
lower threshold 0.56 V (sa 0.49 V below vdd, Vth of XSP2 0.44 V)
hysteresis 0.34 V, margin to the rails 0.60 V above and 0.56 V below


The thresholds are a rough estimate of where the feedback loses against the input stack. They sit well inside the rails, which is all the restorer needs. The README reports that a one-sided ring swinging 0.86 to 1.5 V does not toggle it, which is consistent with a lower threshold well above 0.86 V being out of reach for that swing only if the lower threshold here is in that range. This is checked in the comparison below.

### Pass switch and branch switch

`MSP` passes `ibias_in` to `ibias_pad` while `di_osc_en` is low. Its gate sits at `vss`, its source is the `ibias_in` node and its drain the pad, with the bulk at `VDD` (assumed, as for `MBS` of `bias_gm`). The bank forces the current, so the source rises above the pad voltage `V` until `MSP` carries it: `VSG = Vin`, `VSD = Vin - V`, `VSB = VDD - Vin`. The table gives the current at that bias and a bisection finds `Vin`. `Ron` is the drop over the current. `MSN` is an HV NMOS with its gate at `VDD` and its source on `vg`, treated with `Ron = 1 / GDS` in triode at small `VDS` like the template.

In [12]:
sp, sn = DEV['MSP'], DEV['MSN']
SIM_VIN = {(0, 0.3): 0.477, (4, 0.3): 0.490, (15, 0.3): 0.513,
           (0, 0.5): 0.515, (4, 0.5): 0.521, (15, 0.5): 0.535,
           (0, 0.8): 0.801, (4, 0.8): 0.802, (15, 0.8): 0.803}     # V, tt, README pad table

def msp_vin(i, v):
    f = lambda vin: get(pmos, 'ID', sp['w'] * sp['m'], L=sp['l'], VGS=vin, VDS=vin - v, VSB=VDD - vin) - i
    lo, hi = v + 1e-4, 1.4
    for _ in range(50):
        mid = (lo + hi) / 2
        if f(mid) < 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

print(f"MSP W {sp['w']} um x {sp['m']}, L {sp['l']} um")
print(f"{'pad (V)':>8} {'code':>4} {'Vin (V)':>8} {'drop (mV)':>10} {'Ron (kohm)':>11} {'sim Vin (V)':>12}")
for v in (0.3, 0.5, 0.8):
    for c in CODES:
        i = ibias_in(c)
        vin = msp_vin(i, v)
        print(f"{v:8.1f} {c:4d} {vin:8.3f} {(vin - v)*1e3:10.1f} {(vin - v)/i/1e3:11.0f} {SIM_VIN[(c, v)]:12.3f}")

imax = ibias_in(15)
vg15 = vgs_for(nmos, imax / DEV['MD']['m'])
gn = get(hvn, 'GDS', sn['w'] * sn['m'], L=sn['l'], VGS=VDD - vg15, VDS=0.01, VSB=vg15)
print(f"MSN W {sn['w']} um, L {sn['l']} um, vg {vg15:.2f} V at code 15: Ron {1/gn/1e3:.1f} kohm, drop {imax/gn*1e3:.2f} mV")

MSP W 1.0 um x 2, L 0.5 um
 pad (V) code  Vin (V)  drop (mV)  Ron (kohm)  sim Vin (V)
     0.3    0    0.476      175.5        1317        0.477
     0.3    4    0.488      188.1         941        0.490
     0.3   15    0.513      212.5         555        0.513
     0.5    0    0.517       16.8         126        0.515
     0.5    4    0.523       22.6         113        0.521
     0.5   15    0.537       37.5          98        0.535
     0.8    0    0.801        1.3          10        0.801
     0.8    4    0.802        2.0          10        0.802
     0.8   15    0.804        3.7          10        0.803
MSN W 1.0 um, L 0.45 um, vg 0.25 V at code 15: Ron 5.2 kohm, drop 1.98 mV


The pass switch is the headroom limit of the cell: at a pad voltage of 0.3 V it costs about 0.2 V on `ibias_in`, and the 0.5 V `ibias_in` the bank needs is the result. Above 0.5 V the drop is a few tens of millivolts. The pad error stays small because the bank is a current source, not because `MSP` has a low resistance. `MSN` drops 1 to 2 mV and is not critical. The choice of 1u m2 for `MSP` follows the packing of the layout, and `MSN` is HV for the leakage argument in the README (an LV `MSN` leaked 9 % at ff 125 C).

### Comparison, derived against drawn

In [13]:
rows = [
    ('tail NMOS XTN', 'ideal 1 : 2 of MD', f"W {RING['XTN']['w']} L {RING['XTN']['l']} m {RING['XTN']['m']}", 'unit finger, from layout'),
    ('M2 / MPD / XTP', 'ideal 1 : 2 of MD', f"M2 m {DEV['M2']['m']}, MPD m {DEV['MPD']['m']}, XTP m {RING['XTP']['m']}", 'drawn'),
    ('MD multiplier', f"{md_req:.2f} (model), {DEV['MD']['m']*SIM_TT[4]*1e3/F_TARGET:.2f} (sim scaled)", f"m {DEV['MD']['m']}", 'drawn'),
    ('node cap, mean', f"{C_REQ*1e15:.1f} fF (model), {C_REQ_S*1e15:.1f} fF (sim scaled)", f"{C_AVG*1e15:.1f} fF", f"XC W {xc['w']} L {xc['l']}"),
    ('f_clk code 4', f"{F_TARGET/1e3:.0f} kHz target", f"model {f_mod[4]/1e3:.0f}, sim {SIM_TT[4]:.0f} kHz", ''),
]
for r in rows:
    print(' | '.join(r))

tail NMOS XTN | ideal 1 : 2 of MD | W 0.5 L 0.5 m 1 | unit finger, from layout
M2 / MPD / XTP | ideal 1 : 2 of MD | M2 m 1, MPD m 1, XTP m 1 | drawn
MD multiplier | 1.40 (model), 1.61 (sim scaled) | m 2 | drawn
node cap, mean | 6.5 fF (model), 7.5 fF (sim scaled) | 9.3 fF | XC W 0.5 L 0.5
f_clk code 4 | 1024 kHz target | model 715, sim 823 kHz | 


### Result

Disagreements between the derivation and the drawn cell, nothing was changed:

1. The model puts code 4 at 715 kHz with ideal mirror currents for the drawn sizes, 30 % below the 1.024 MHz target. The simulation is 20 % below. No cap size reaches the target in the model, because the part of the node that is not `XC` is already 6.7 fF against 6.5 fF required, and `MD` m2 is the only whole multiplier that is on the slow side of the target (m1 doubles the current).
2. `n5` carries 16.1 fF against 7.6 fF for `n1` to `n4`, because the four Schmitt input gates (8.5 fF) hang on it. This is why the node cap is large and the `XC` caps cannot be shrunk much further for speed. It does not change the ring period in this model, which only depends on the sum.
3. The simulated frequency is less than linear in the bank current, which this model does not show.
4. The Schmitt trigger estimate puts the lower threshold at about 0.56 V. A ring starved on the NMOS side only swings 0.86 to 1.5 V (README, simulation), so it never crosses that threshold, which agrees with the README that this variant sticks.
5. The pass switch costs about 0.2 V of headroom on `ibias_in` at a pad voltage of 0.3 V, which the pad table in the README shows too (0.477 to 0.513 V, simulation). The pass switch is sized by the layout, not by the resistance.

The code 4 miss is accepted for a trimmed test clock and `MD` m2 stays as drawn.